In [1]:
suppressMessages(library("rwwa"))

fpath = "~/00_WWA_project_folder/ephemeral/worldcup/wbgt_maxs/"

In [2]:
stadia <- c("Atlanta", "Boston", "Dallas", "Guadalajara", "Houston", "Los_Angeles", "Mexico", "Monterrey", "Toronto", "Vancouver", "Miami", "Seattle", "Kansas", "San_Francisco", "New_York", "Philadelphia")

# Loop over models and estimate trend with low RP

In [39]:
for (stadium in stadia) {
    for (fnm in list.files(paste0(fpath, "models/wbgt_maxs"), pattern = stadium, full.names = T)) {
        
        t0 <- Sys.time()

        gcm <- gsub(".+/|_.+", "", fnm)
        cat(stadium, gcm)
        
        res_fnm <- paste0("res/res_mdl_wbgt1x_",gcm,"_",stadium,".csv")
        if (file.exists(res_fnm)) {print("already exists"); next}
        
        gmst_fnm <- list.files("~/00_WWA_project_folder/live/data/cmip6-gmsts/", paste0(gcm,"_"), full.names = T)
    
        if(length(gmst_fnm) == 0) {print("no GMST"); next}
        if(length(gmst_fnm) > 1) gmst_fnm <- gmst_fnm[1]
    
        ts <- read.csv(fnm)[,c("year", "wbgt")]
        gmst <- load_ts(gmst_fnm, col.names = c("year", "gmst"))
        df <- merge(ts, gmst)
    
        # trend fitting
        mdl <- fit_ns("gev", "shift", data = df, varnm = "wbgt", covnm = "gmst", lower = F, method = "Nelder-Mead")
        
        # use model's 2023 GMST & observed Nino to define factual climate
        gmst_2025 <- df$gmst[df$year == 2026]
        
        cov_2025 <- data.frame("gmst" = gmst_2025)
        cov_cf <- data.frame("gmst" = gmst_2025 - 0.7)
        cov_fut <- data.frame("gmst" = gmst_2025 + 0.7)
    
        # bootstrap results
        res <- cmodel_results(mdl, rp = , cov_f = cov_2025, cov_hist = cov_cf, cov_fut = cov_fut,
                              y_now = 2026, y_start = 1979, y_fut = 2070, nsamp = 500)
    
        write.csv(res, res_fnm)
        t1 <- Sys.time()
        print(t1 - t0)
    }
}
print("done.")

Atlanta CNRM-CM6-1-HR[1] "already exists"
Atlanta GFDL-CM4[1] "already exists"
Atlanta HadGEM3-GC31-MM[1] "already exists"
Atlanta INM-CM5-0[1] "already exists"
Atlanta NorESM2-MM[1] "already exists"
Boston CNRM-CM6-1-HR[1] "already exists"
Boston GFDL-CM4[1] "already exists"
Boston HadGEM3-GC31-MM[1] "already exists"
Boston INM-CM5-0[1] "already exists"
Boston NorESM2-MM[1] "already exists"
Dallas CNRM-CM6-1-HR[1] "already exists"
Dallas GFDL-CM4[1] "already exists"
Dallas HadGEM3-GC31-MM[1] "already exists"
Dallas INM-CM5-0[1] "already exists"
Dallas NorESM2-MM[1] "already exists"
Guadalajara CNRM-CM6-1-HR[1] "already exists"
Guadalajara GFDL-CM4[1] "already exists"
Guadalajara HadGEM3-GC31-MM[1] "already exists"
Guadalajara INM-CM5-0[1] "already exists"
Guadalajara NorESM2-MM[1] "already exists"
Houston CNRM-CM6-1-HR[1] "already exists"
Houston GFDL-CM4[1] "already exists"
Houston HadGEM3-GC31-MM[1] "already exists"
Houston INM-CM5-0[1] "already exists"
Houston NorESM2-MM[1] "alread

# Loop over obs and estimate trend and RP

In [29]:
gmst <- read.table("gmst.txt", col.names = c("year", "gmst"))
cov_f <- gmst[gmst$year == 2025,"gmst",drop = F]
cov_cf <- cov_f - 0.7

for (fnm in list.files(paste0(fpath, "obs/wbgt_maxs"),full.names = T)) {

    stadium <- gsub(".+/|_.+","",fnm)
    res_fnm <- paste0("res/res-obs_wbgt_",stadium,".csv")
    if(file.exists(res_fnm)) next
    
    ts <- read.csv(fnm)[,c("year", "wbgt")]
    df <- merge(gmst, ts)

    mdl <- fit_ns("gev", "shift", df, varnm = "wbgt", covnm <- "gmst", lower = F)

    res <- boot_ci(mdl, cov_f = cov_f, cov_cf = cov_cf, nsamp = 1000)
    write.csv(res, res_fnm)
}

## Compile obs

In [38]:
list.files("res/res-obs")

character(0)

# Results with threshold RP

## Modified functions

In [3]:
th_ests <- function(mdl, cov_f, cov_cf, thresholds = c(26,28,30,32), rps) {
    pars <- mdl$par
    current_pars <- ns_pars(mdl, fixed_cov = cov_f)

    th_rp <- setNames(sapply(thresholds, function(th) return_period(mdl = mdl, x = th, fixed_cov = cov_f)), paste0("rp_",thresholds))
    th_pr <- setNames(sapply(thresholds, function(th) prob_ratio(mdl, th, cov_f, cov_cf)), paste0("pr_",thresholds))
    
    dI <- setNames(int_change(mdl, 5, cov_f, cov_cf), "dI_abs")

    res <- 

    if (missing(rps)) {
        return(c(mdl$par, dI, th_rp, th_pr, aic = aic(mdl)))
    } else {
        rls <- setNames(sapply(rps, function(rp) if(is.finite(rp)) {eff_return_level(mdl, rp, fixed_cov = cov_f)} else {NA}), paste0("rl_",thresholds))
        return(c(mdl$par, dI, th_rp, th_pr, rls, aic = aic(mdl)))
    }
}

In [4]:
# modified function to bootstrap return periods for all thresholds
boot_th <- function(mdl, cov_f, cov_cf, seed = 42, nsamp = 500, ci = 0.95, return_sample = F, thresholds = c(26,28,30,32), rps) {
    alpha <- 1 - ci

    cov_f <- cov_f[, mdl$covnm, drop = F]
    cov_cf <- cov_cf[, mdl$covnm, drop = F]

    obs_res <- th_ests(mdl, cov_f, cov_cf, thresholds = thresholds, rps)
    set.seed(seed)
    boot_res <- list()
    i <- 1
    f <- 0
    while (length(boot_res) < nsamp) {
        boot_df <- mdl$data[sample(1:nrow(mdl$data), replace = T), ]
        tryCatch({
            boot_mdl <- refit(mdl, new_data = boot_df)
            boot_res[[i]] <- th_ests(boot_mdl, cov_f, cov_cf, thresholds = thresholds, rps)
            i <- i + 1
        }, error = function(cond) {
            f <- f + 1
            return(NULL)
        })
    }
    boot_res <- do.call("cbind", boot_res)
    if (return_sample) {
        return(boot_res)
    } else {
        boot_qq <- t(rbind(est = obs_res, apply(boot_res, 1, quantile, c(alpha/2, 0.5, 1 - (alpha/2)), na.rm = T)))
        return(boot_qq)
    }
}

In [5]:
cmodel_th <- function (mdl, th_rp, cov_f, cov_hist, cov_fut, y_start = 1979, y_now = 2026, y_fut = 2070, nsamp = 5, seed = 42) {
    
    set.seed(seed)
    cov_f <- cov_f[, mdl$covnm, drop = F]
    cov_hist <- cov_hist[, mdl$covnm, drop = F]
    if (nrow(cov_f) > 1) {
        print("cov_f has more than one row: only first row will be used as factual covariates")
        cov_f <- cov_f[1, , drop = F]
    }
    df <- mdl$data
    mdl_eval <- refit(mdl, new_data = df[df$year >= y_start & df$year <= y_now, ])
    mdl_attr <- refit(mdl, new_data = df[df$year <= y_now, ])
    mdl_proj <- refit(mdl, new_data = df[df$year <= y_fut, ])

    th_rl <- sapply(th_rp, function(rp) if(!is.finite(rp)) { NA } else { eff_return_level(mdl_attr, rp, fixed_cov = cov_2025) })
    th_rl <- setNames(th_rl, paste0("rp_", c(26,28,30,32)))

    # fit the different types of models in turn
    ci_eval <- boot_th(mdl_eval, cov_f = cov_f, cov_cf = cov_hist, nsamp = nsamp, thresholds = th_rl, rps = th_rp)                    
                        
    ci_attr <- boot_th(mdl_attr, cov_f = cov_f, cov_cf = cov_hist, nsamp = nsamp, thresholds = th_rl, rps = th_rp)
    rownames(ci_attr)[grepl("rp", rownames(ci_attr))] <- paste0("rp", c(26,28,30,32))
    rownames(ci_attr)[grepl("pr", rownames(ci_attr))] <- paste0("PR", c(26,28,30,32))
    rownames(ci_attr)[grepl("rl", rownames(ci_attr))] <- paste0("rl", c(26,28,30,32))
      
    ci_proj <- boot_th(mdl_proj, cov_f = cov_f, cov_cf = cov_hist, nsamp = nsamp, thresholds = th_rl, rps = th_rp)
    rownames(ci_proj)[grepl("rp", rownames(ci_proj))] <- paste0("rp", c(26,28,30,32))
    rownames(ci_proj)[grepl("pr", rownames(ci_proj))] <- paste0("PR", c(26,28,30,32))
    rownames(ci_proj)[grepl("rl", rownames(ci_proj))] <- paste0("rl", c(26,28,30,32))

    # extract rows & compile
    ci_eval <- ci_eval[c("sigma0", "alpha_gmst", "shape"),]
    ci_attr <- ci_attr[grepl("dI_abs|PR|rp|rl", rownames(ci_attr)), ]
    ci_proj <- ci_proj[grepl("dI_abs|PR|rp|rl", rownames(ci_proj)), ]
            
    ci_eval <- unlist(lapply(rownames(ci_eval), function(cnm) setNames(ci_eval[cnm, ], paste("eval", gsub("_", "-", cnm), c("est", "lower", "med", "upper"), sep = "_"))))
    ci_attr <- unlist(lapply(rownames(ci_attr), function(cnm) setNames(ci_attr[cnm, ], paste("attr", gsub("_", "-", cnm), c("est", "lower", "med", "upper"), sep = "_"))))
    ci_proj <- unlist(lapply(rownames(ci_proj), function(cnm) setNames(ci_proj[cnm, ], paste("proj", gsub("_", "-", cnm), c("est", "lower", "med", "upper"), sep = "_"))))

    res <- t(data.frame(c(ci_eval, ci_attr, ci_proj)))
    rownames(res) <- "wbgt ~ gmst (th)"
    return(res)
}                                 

## Obs

In [328]:
gmst <- read.table("gmst.txt", col.names = c("year", "gmst"))
cov_f <- gmst[gmst$year == 2025,"gmst",drop = F]
cov_cf <- cov_f - 0.7

for (fnm in list.files(paste0(fpath, "obs/wbgt_maxs"), full.names = T)) {

    stadium <- gsub("_.+","",gsub(".+/","",fnm))
    stadium <- switch(stadium, "Los" = "Los-Angeles", "New" = "New-York", "San" = "San-Francisco", "Mexico" = "Mexico-City", stadium)
    res_fnm <- paste0("res_vrp/res-obs_vrp_",stadium,".csv")
    if (file.exists(res_fnm)) next
    
    ts <- read.csv(fnm)[,c("year", "wbgt")]
    df <- merge(gmst, ts)

    mdl <- fit_ns("gev", "shift", df, covnm = "gmst", varnm = "wbgt", lower = F)
    boot_ci <- boot_th(mdl, cov_f, cov_cf, nsamp = 1000)

    write.csv(boot_ci, res_fnm)
}

## Climate models

In [10]:
stadia <- c("Atlanta", "Boston", "Dallas", "Guadalajara", "Houston", "Los_Angeles", "Mexico", "Monterrey", "Toronto", "Vancouver", "Miami", "Seattle", "Kansas", "San_Francisco", "New_York", "Philadelphia")

for (stadium in stadia) {
    for (fnm in list.files(paste0(fpath, "models/wbgt_maxs"), pattern = stadium, full.names = T)) {
    
        gcm <- gsub("_.+","",gsub(".+/","",fnm))
        stadium <- gsub("_.+","",gsub(paste0(".+",gcm,"_"), "",fnm))
        stadium <- switch(stadium, "Los" = "Los-Angeles", "New" = "New-York", "San" = "San-Francisco", "Mexico" = "Mexico-City", stadium)
        
        res_fnm <- paste0("res_vrp/res-mdl_vrp_",stadium,"_",gcm,".csv")
        if (file.exists(res_fnm)) next
        
        gmst_fnm <- list.files("~/00_WWA_project_folder/live/data/cmip6-gmsts/", paste0(gcm,"_"), full.names = T)
        
        if(length(gmst_fnm) == 0) {print("no GMST"); next}
        if(length(gmst_fnm) > 1) gmst_fnm <- gmst_fnm[1]
    
        # get observed return periods
        obs_res <- read.csv(list.files("res_vrp", paste0("res-obs_vrp_", stadium), full.names = T), row.names = "X")
        th_rp <- obs_res[grepl("rp", rownames(obs_res)),"est"]
        th_rp <- th_rp + ((th_rp == 1) * 10e-15) # to avoid issues with rp == 1
        
        ts <- read.csv(fnm)[,c("year", "wbgt")]
        gmst <- load_ts(gmst_fnm, col.names = c("year", "gmst"))
        df <- merge(ts, gmst)
    
        cov_2025 <- df[df$year == 2026,"gmst",drop = F]
        cov_1994 <- cov_2025 - 0.7
        cov_fut <- cov_2025 + 0.7
    
        mdl <- fit_ns("gev", "shift", df, covnm = "gmst", varnm = "wbgt", lower = F, method = "Nelder-Mead")
        boot_res <- cmodel_th(mdl = mdl, th_rp = th_rp, cov_f = cov_2025, cov_hist = cov_1994, cov_fut = cov_fut, y_start = 1979, y_now = 2026, y_fut = 2050, nsamp = 1000, seed = 42)
        rownames(boot_res) <- paste0(stadium,"_",gcm)
    
        write.csv(boot_res, res_fnm)
    }
}
print("All done.")

[1] "All done."


### Compile all results

In [ ]:
# models
df <- do.call("rbind", sapply(list.files("res_vrp", pattern = "res-mdl", full.names = T), read.csv, simplify = F))
rownames(df) <- df$X
df <- df[,-1]
write.csv(df, "res-mdl_vrp.csv")

In [ ]:
# obs
z <- do.call("rbind", sapply(list.files("res_vrp", pattern = "res-obs", full.names = T), function(fnm) {
    df <- read.csv(fnm, row.names = "X")
    res <- unlist(sapply(rownames(df), function(rnm) {setNames(unlist(df[rnm,]), c("est", "lower", "med", "upper"))}, simplify = F))
    res
}, simplify = F))
rownames(z) <- gsub(".+_|.csv","",rownames(z))
write.csv(z, "res-obs_vrp.csv")